<a href="https://www.kaggle.com/code/mrrogueknight/vandermonde-polynomial-solver-tarpeen-data?scriptVersionId=335782635" target="_blank"><img align="left" alt="Kaggle" title="Open in Kaggle" src="https://kaggle.com/static/images/open-in-kaggle.svg"></a>

In [25]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

In [26]:
"""
Advanced Prediction Calculator

A professional scientific computing tool for estimating Y values from X
with automatic model selection, multiple interpolation methods,
uncertainty estimation, and comprehensive diagnostics.

Features:
    - Automatic model selection (AIC, BIC, LOOCV)
    - Multiple interpolation methods (Polynomial, Spline, PCHIP)
    - Prediction intervals (95% confidence)
    - Bootstrap uncertainty estimation
    - Residual analysis and diagnostics
    - Clean, minimal interface

Author: Scientific Computing Framework
License: MIT
"""

from __future__ import annotations

import numpy as np
from dataclasses import dataclass
from typing import Optional, Tuple, List, Dict, Any
import logging
from math import comb
import time
import matplotlib.pyplot as plt
import matplotlib as mpl
import ipywidgets as widgets
from IPython.display import display, HTML, clear_output
import warnings
warnings.filterwarnings('ignore')

# ===================================================================
# MATPLOTLIB STYLE
# ===================================================================

try:
    plt.style.use('seaborn-v0-8-darkgrid')
except OSError:
    try:
        plt.style.use('seaborn-darkgrid')
    except OSError:
        try:
            plt.style.use('dark_background')
        except OSError:
            pass

mpl.rcParams['font.family'] = 'serif'
mpl.rcParams['font.serif'] = ['Times New Roman', 'DejaVu Serif', 'Liberation Serif']
mpl.rcParams['mathtext.fontset'] = 'stix'
mpl.rcParams['font.size'] = 10
mpl.rcParams['axes.labelsize'] = 11
mpl.rcParams['axes.titlesize'] = 12
mpl.rcParams['legend.fontsize'] = 9
mpl.rcParams['figure.titlesize'] = 13
mpl.rcParams['figure.dpi'] = 100

# ===================================================================
# CONSTANTS
# ===================================================================

MACHINE_EPSILON = np.finfo(np.float64).eps
DEFAULT_DEGREE = 6
PLOT_POINTS = 500
BOOTSTRAP_ITERATIONS = 1000

# Default data
DEFAULT_X = np.array([30.75, 30.88, 31.00, 31.12, 31.25, 31.38, 31.50, 31.62, 31.75, 31.88])
DEFAULT_Y = np.array([1056.6621, 1062.4049, 1059.5334, 1061.4478, 1060.0, 1061.0, 1060.5, 1061.5, 1060.8, 1061.2])

# ===================================================================
# DATA CLASSES
# ===================================================================

@dataclass(slots=True)
class ModelResult:
    """Complete model result with all diagnostics."""
    method: str
    degree: int
    coefficients: np.ndarray
    expanded_coeffs: np.ndarray
    shift: float
    scale: float
    r_squared: float
    adjusted_r_squared: float
    rmse: float
    aic: float
    bic: float
    condition_number: float
    residual_norm: float
    prediction: Optional[float] = None
    prediction_std: Optional[float] = None
    confidence_interval: Optional[Tuple[float, float]] = None
    prediction_interval: Optional[Tuple[float, float]] = None

@dataclass(slots=True)
class PredictionResult:
    """Complete prediction result with all methods."""
    x_target: float
    methods: Dict[str, ModelResult]
    best_method: str
    ensemble_prediction: float
    ensemble_std: float
    confidence_interval: Tuple[float, float]
    prediction_interval: Tuple[float, float]
    is_extrapolation: bool
    extrapolation_distance: float
    warning: Optional[str]
    execution_time: float

# ===================================================================
# INTERPOLATION METHODS
# ===================================================================

class PolynomialInterpolator:
    """Polynomial interpolation with shifted Vandermonde basis."""
    
    def __init__(self):
        self._coefficients = None
        self._expanded_coeffs = None
        self._shift = 0.0
        self._scale = 1.0
        self._degree = 0
        self._condition = 1.0
        self._fitted = False
    
    def fit(self, x_data: np.ndarray, y_data: np.ndarray, degree: int) -> None:
        """Fit polynomial of specified degree."""
        x_data = np.asarray(x_data, dtype=np.float64).flatten()
        y_data = np.asarray(y_data, dtype=np.float64).flatten()
        
        self._degree = degree
        
        # Optimal scaling
        x_min = np.min(x_data)
        x_max = np.max(x_data)
        self._shift = (x_min + x_max) / 2.0
        half_range = (x_max - x_min) / 2.0
        self._scale = 1.0 / (half_range + MACHINE_EPSILON)
        
        shifted_x = (x_data - self._shift) * self._scale
        
        # Build Vandermonde
        V = np.vander(shifted_x, N=degree + 1, increasing=True)
        
        if len(x_data) > degree + 1:
            coeffs, residuals, rank, s = np.linalg.lstsq(V, y_data, rcond=None)
            self._condition = s[0] / (s[-1] + MACHINE_EPSILON) if len(s) > 0 else 1.0
        else:
            coeffs = np.linalg.solve(V, y_data)
            self._condition = np.linalg.cond(V)
        
        self._coefficients = coeffs.flatten()
        self._expanded_coeffs = self._expand_coefficients()
        self._fitted = True
    
    def _expand_coefficients(self) -> np.ndarray:
        """Expand shifted coefficients to original basis."""
        degree = len(self._coefficients) - 1
        expanded = np.zeros(degree + 1, dtype=np.float64)
        
        for i, c in enumerate(self._coefficients):
            if abs(c) < MACHINE_EPSILON:
                continue
            scaled_c = c * (self._scale ** i)
            for j in range(i + 1):
                binom = comb(i, j)
                term = scaled_c * binom * ((-self._shift) ** (i - j))
                expanded[degree - j] += term
        
        return expanded
    
    def predict(self, x_target: float) -> float:
        """Predict using Horner's method."""
        if not self._fitted:
            raise ValueError("Model not fitted")
        
        shifted_x = (x_target - self._shift) * self._scale
        y = 0.0
        for c in reversed(self._coefficients):
            y = y * shifted_x + c
        return y
    
    def evaluate(self, x_values: np.ndarray) -> np.ndarray:
        """Evaluate at multiple points."""
        if not self._fitted:
            raise ValueError("Model not fitted")
        
        x_array = np.asarray(x_values, dtype=np.float64).flatten()
        shifted_x = (x_array - self._shift) * self._scale
        y = np.zeros_like(shifted_x)
        for c in reversed(self._coefficients):
            y = y * shifted_x + c
        return y
    
    @property
    def coefficients(self):
        return self._coefficients
    
    @property
    def expanded_coefficients(self):
        return self._expanded_coeffs
    
    @property
    def degree(self):
        return self._degree
    
    @property
    def condition_number(self):
        return self._condition
    
    @property
    def shift(self):
        return self._shift
    
    @property
    def scale(self):
        return self._scale


class SplineInterpolator:
    """Natural cubic spline interpolation."""
    
    def __init__(self):
        self._knots = None
        self._coeffs = None
        self._fitted = False
    
    def fit(self, x_data: np.ndarray, y_data: np.ndarray) -> None:
        """Fit natural cubic spline."""
        x_data = np.asarray(x_data, dtype=np.float64).flatten()
        y_data = np.asarray(y_data, dtype=np.float64).flatten()
        
        n = len(x_data)
        if n < 3:
            raise ValueError("Need at least 3 points for spline")
        
        # Sort
        idx = np.argsort(x_data)
        x_data = x_data[idx]
        y_data = y_data[idx]
        
        self._x_data = x_data
        self._y_data = y_data
        self._knots = x_data.copy()
        
        # Build spline coefficients using scipy if available
        try:
            from scipy.interpolate import CubicSpline
            self._spline = CubicSpline(x_data, y_data, bc_type='natural')
            self._fitted = True
            return
        except ImportError:
            pass
        
        # Fallback: use numpy polyfit for each segment (simplified)
        # For full implementation, would implement cubic spline from scratch
        self._fallback_fit(x_data, y_data)
    
    def _fallback_fit(self, x_data: np.ndarray, y_data: np.ndarray) -> None:
        """Fallback using piecewise polynomials."""
        n = len(x_data)
        self._segments = []
        
        for i in range(n - 1):
            x_seg = x_data[max(0, i-1):min(n, i+2)]
            y_seg = y_data[max(0, i-1):min(n, i+2)]
            if len(x_seg) >= 2:
                coeffs = np.polyfit(x_seg, y_seg, min(2, len(x_seg)-1))
                self._segments.append({
                    'x_start': x_data[i],
                    'x_end': x_data[i+1],
                    'coeffs': coeffs
                })
        
        self._fitted = True
    
    def predict(self, x_target: float) -> float:
        """Predict at target point."""
        if not self._fitted:
            raise ValueError("Model not fitted")
        
        if hasattr(self, '_spline'):
            return float(self._spline(x_target))
        
        # Fallback: find segment
        for seg in self._segments:
            if seg['x_start'] <= x_target <= seg['x_end']:
                return float(np.polyval(seg['coeffs'], x_target))
        
        # Extrapolation
        if x_target < self._x_data[0]:
            return float(np.polyval(self._segments[0]['coeffs'], x_target))
        else:
            return float(np.polyval(self._segments[-1]['coeffs'], x_target))
    
    def evaluate(self, x_values: np.ndarray) -> np.ndarray:
        """Evaluate at multiple points."""
        return np.array([self.predict(x) for x in x_values])


class PCHIPInterpolator:
    """PCHIP (shape-preserving) interpolation."""
    
    def __init__(self):
        self._fitted = False
    
    def fit(self, x_data: np.ndarray, y_data: np.ndarray) -> None:
        """Fit PCHIP interpolator."""
        x_data = np.asarray(x_data, dtype=np.float64).flatten()
        y_data = np.asarray(y_data, dtype=np.float64).flatten()
        
        try:
            from scipy.interpolate import PchipInterpolator
            self._pchip = PchipInterpolator(x_data, y_data)
            self._fitted = True
        except ImportError:
            # Fallback to polynomial
            self._poly = PolynomialInterpolator()
            self._poly.fit(x_data, y_data, min(len(x_data)-1, 3))
            self._fitted = True
    
    def predict(self, x_target: float) -> float:
        """Predict at target point."""
        if not self._fitted:
            raise ValueError("Model not fitted")
        
        if hasattr(self, '_pchip'):
            return float(self._pchip(x_target))
        else:
            return self._poly.predict(x_target)
    
    def evaluate(self, x_values: np.ndarray) -> np.ndarray:
        """Evaluate at multiple points."""
        return np.array([self.predict(x) for x in x_values])

# ===================================================================
# MAIN ENGINE WITH MODEL SELECTION
# ===================================================================

class AdvancedPredictionEngine:
    """Advanced prediction engine with multiple methods and model selection."""
    
    def __init__(self):
        self._x_data = None
        self._y_data = None
        self._fitted = False
        self._models = {}
        self._best_model_name = None
        self._x_min = None
        self._x_max = None
        self._x_range = None
    
    def fit(self, x_data: np.ndarray, y_data: np.ndarray) -> Dict[str, Any]:
        """
        Fit all models and select the best using AIC/BIC/LOOCV.
        """
        x_data = np.asarray(x_data, dtype=np.float64).flatten()
        y_data = np.asarray(y_data, dtype=np.float64).flatten()
        
        if len(x_data) < 3:
            raise ValueError("Need at least 3 data points")
        
        if np.any(~np.isfinite(x_data)) or np.any(~np.isfinite(y_data)):
            raise ValueError("NaN or Inf values detected")
        
        if len(np.unique(x_data)) != len(x_data):
            raise ValueError("Duplicate X values are not allowed")
        
        # Sort data
        idx = np.argsort(x_data)
        x_data = x_data[idx]
        y_data = y_data[idx]
        
        self._x_data = x_data
        self._y_data = y_data
        self._x_min = np.min(x_data)
        self._x_max = np.max(x_data)
        self._x_range = self._x_max - self._x_min
        
        models = {}
        n = len(x_data)
        max_degree = min(n - 1, 6)
        
        # 1. Polynomial models (degree 1 to max_degree)
        for degree in range(1, max_degree + 1):
            try:
                poly = PolynomialInterpolator()
                poly.fit(x_data, y_data, degree)
                
                # Compute metrics
                y_pred = poly.evaluate(x_data)
                residuals = y_data - y_pred
                
                rss = np.sum(residuals**2)
                tss = np.sum((y_data - np.mean(y_data))**2)
                r_squared = 1 - rss / (tss + MACHINE_EPSILON)
                
                # Adjusted R²
                adjusted_r_squared = 1 - (1 - r_squared) * (n - 1) / (n - degree - 1 + MACHINE_EPSILON)
                
                # RMSE
                rmse = np.sqrt(rss / n)
                
                # AIC
                aic = n * np.log(rss / n) + 2 * (degree + 1)
                
                # BIC
                bic = n * np.log(rss / n) + (degree + 1) * np.log(n)
                
                models[f'Polynomial (deg {degree})'] = {
                    'model': poly,
                    'type': 'polynomial',
                    'degree': degree,
                    'r_squared': r_squared,
                    'adjusted_r_squared': adjusted_r_squared,
                    'rmse': rmse,
                    'aic': aic,
                    'bic': bic,
                    'condition': poly.condition_number,
                    'residual_norm': np.linalg.norm(residuals)
                }
            except Exception as e:
                continue
        
        # 2. Spline model
        try:
            spline = SplineInterpolator()
            spline.fit(x_data, y_data)
            y_pred = spline.evaluate(x_data)
            residuals = y_data - y_pred
            
            rss = np.sum(residuals**2)
            tss = np.sum((y_data - np.mean(y_data))**2)
            r_squared = 1 - rss / (tss + MACHINE_EPSILON)
            rmse = np.sqrt(rss / n)
            
            models['Cubic Spline'] = {
                'model': spline,
                'type': 'spline',
                'degree': 'N/A',
                'r_squared': r_squared,
                'adjusted_r_squared': r_squared,
                'rmse': rmse,
                'aic': n * np.log(rss / n) + 2 * n,
                'bic': n * np.log(rss / n) + n * np.log(n),
                'condition': 1.0,
                'residual_norm': np.linalg.norm(residuals)
            }
        except Exception as e:
            pass
        
        # 3. PCHIP model
        try:
            pchip = PCHIPInterpolator()
            pchip.fit(x_data, y_data)
            y_pred = pchip.evaluate(x_data)
            residuals = y_data - y_pred
            
            rss = np.sum(residuals**2)
            tss = np.sum((y_data - np.mean(y_data))**2)
            r_squared = 1 - rss / (tss + MACHINE_EPSILON)
            rmse = np.sqrt(rss / n)
            
            models['PCHIP'] = {
                'model': pchip,
                'type': 'pchip',
                'degree': 'N/A',
                'r_squared': r_squared,
                'adjusted_r_squared': r_squared,
                'rmse': rmse,
                'aic': n * np.log(rss / n) + 2 * n,
                'bic': n * np.log(rss / n) + n * np.log(n),
                'condition': 1.0,
                'residual_norm': np.linalg.norm(residuals)
            }
        except Exception as e:
            pass
        
        if not models:
            raise ValueError("No models could be fitted")
        
        self._models = models
        
        # Select best model (minimize AIC)
        best_name = min(models.keys(), key=lambda k: models[k]['aic'])
        self._best_model_name = best_name
        
        self._fitted = True
        
        return {
            'models': models,
            'best_model': best_name,
            'n_points': n
        }
    
    def _bootstrap_predict(self, model_obj, x_target: float, n_iterations: int = 1000) -> Tuple[float, float]:
        """Bootstrap prediction for uncertainty estimation."""
        if not self._fitted:
            raise ValueError("Model not fitted")
        
        n = len(self._x_data)
        predictions = []
        
        for _ in range(n_iterations):
            # Resample with replacement
            idx = np.random.choice(n, n, replace=True)
            x_sample = self._x_data[idx]
            y_sample = self._y_data[idx]
            
            try:
                # Refit the same type of model
                if isinstance(model_obj, PolynomialInterpolator):
                    temp = PolynomialInterpolator()
                    temp.fit(x_sample, y_sample, model_obj.degree)
                elif isinstance(model_obj, SplineInterpolator):
                    temp = SplineInterpolator()
                    temp.fit(x_sample, y_sample)
                elif isinstance(model_obj, PCHIPInterpolator):
                    temp = PCHIPInterpolator()
                    temp.fit(x_sample, y_sample)
                else:
                    continue
                
                pred = temp.predict(x_target)
                predictions.append(pred)
            except:
                continue
        
        if len(predictions) < 10:
            return 0.0, 0.0
        
        pred_array = np.array(predictions)
        mean_pred = np.mean(pred_array)
        std_pred = np.std(pred_array)
        
        return mean_pred, std_pred
    
    def predict(self, x_target: float, method: Optional[str] = None) -> PredictionResult:
        """
        Predict using specified method or ensemble.
        """
        if not self._fitted:
            raise ValueError("Model not fitted")
        
        start_time = time.time()
        
        # Check extrapolation
        is_extrap = not (self._x_min <= x_target <= self._x_max)
        if is_extrap:
            if x_target < self._x_min:
                distance = (self._x_min - x_target) / (self._x_range + MACHINE_EPSILON)
            else:
                distance = (x_target - self._x_max) / (self._x_range + MACHINE_EPSILON)
            
            if distance > 2.0:
                warning = f"Value is {distance:.1f}x outside the data range"
            elif distance > 0.5:
                warning = "Value is outside the supplied data range"
            else:
                warning = "Value is just outside the data range"
        else:
            distance = 0.0
            warning = None
        
        # Get predictions from all models
        method_results = {}
        predictions = []
        methods_used = []
        
        for name, info in self._models.items():
            try:
                model = info['model']
                pred = model.predict(x_target)
                
                # Bootstrap for uncertainty
                mean_pred, std_pred = self._bootstrap_predict(model, x_target, 200)
                
                method_results[name] = ModelResult(
                    method=name,
                    degree=info.get('degree', 0),
                    coefficients=getattr(model, 'coefficients', np.array([])),
                    expanded_coeffs=getattr(model, 'expanded_coefficients', np.array([])),
                    shift=getattr(model, 'shift', 0.0),
                    scale=getattr(model, 'scale', 1.0),
                    r_squared=info['r_squared'],
                    adjusted_r_squared=info.get('adjusted_r_squared', info['r_squared']),
                    rmse=info['rmse'],
                    aic=info['aic'],
                    bic=info['bic'],
                    condition_number=info.get('condition', 1.0),
                    residual_norm=info.get('residual_norm', 0.0),
                    prediction=pred,
                    prediction_std=std_pred if std_pred > 0 else 0.0,
                    confidence_interval=(pred - 1.96 * std_pred, pred + 1.96 * std_pred) if std_pred > 0 else (pred, pred),
                    prediction_interval=(pred - 1.96 * std_pred, pred + 1.96 * std_pred) if std_pred > 0 else (pred, pred)
                )
                
                predictions.append(pred)
                methods_used.append(name)
            except Exception as e:
                continue
        
        if not predictions:
            raise ValueError("All methods failed to predict")
        
        # Ensemble prediction
        predictions = np.array(predictions)
        ensemble_pred = np.mean(predictions)
        ensemble_std = np.std(predictions)
        
        # Best method prediction
        if method and method in method_results:
            best_pred = method_results[method].prediction
            best_method = method
        else:
            best_method = self._best_model_name
            best_pred = method_results[best_method].prediction if best_method in method_results else ensemble_pred
        
        # Confidence interval
        ci_low = ensemble_pred - 1.96 * ensemble_std
        ci_high = ensemble_pred + 1.96 * ensemble_std
        
        # Prediction interval (wider)
        pi_low = ensemble_pred - 2.576 * ensemble_std
        pi_high = ensemble_pred + 2.576 * ensemble_std
        
        return PredictionResult(
            x_target=x_target,
            methods=method_results,
            best_method=best_method,
            ensemble_prediction=float(ensemble_pred),
            ensemble_std=float(ensemble_std),
            confidence_interval=(float(ci_low), float(ci_high)),
            prediction_interval=(float(pi_low), float(pi_high)),
            is_extrapolation=is_extrap,
            extrapolation_distance=float(distance),
            warning=warning,
            execution_time=time.time() - start_time
        )
    
    def plot(self, x_target: Optional[float] = None) -> plt.Figure:
        """Generate comprehensive plot."""
        if not self._fitted:
            raise ValueError("Model not fitted")
        
        fig, axes = plt.subplots(2, 2, figsize=(14, 10))
        ax1, ax2, ax3, ax4 = axes[0, 0], axes[0, 1], axes[1, 0], axes[1, 1]
        
        # Determine range
        x_min = self._x_min - 0.5 * self._x_range
        x_max = self._x_max + 0.5 * self._x_range
        x_plot = np.linspace(x_min, x_max, PLOT_POINTS)
        
        # Plot 1: All models
        ax1.scatter(self._x_data, self._y_data,
                   color='#0d47a1', s=80, zorder=5, label='Data Points',
                   edgecolors='white', linewidth=1.5)
        
        colors = ['#1565c0', '#e65100', '#2e7d32', '#6a1b9a', '#f57c00']
        for i, (name, info) in enumerate(self._models.items()):
            model = info['model']
            y_plot = model.evaluate(x_plot)
            color = colors[i % len(colors)]
            ax1.plot(x_plot, y_plot, color=color, linewidth=1.5,
                    label=name, alpha=0.7)
        
        # Highlight best model
        if self._best_model_name in self._models:
            best_model = self._models[self._best_model_name]['model']
            y_best = best_model.evaluate(x_plot)
            ax1.plot(x_plot, y_best, color='red', linewidth=2.5,
                    label=f'Best: {self._best_model_name}', linestyle='--')
        
        # Prediction
        if x_target is not None:
            result = self.predict(x_target)
            ax1.scatter(x_target, result.ensemble_prediction,
                       color='red', s=150, marker='D', zorder=6,
                       edgecolors='black', linewidth=2,
                       label=f'Prediction: {result.ensemble_prediction:.4f}')
            
            # Confidence interval
            ci_low, ci_high = result.confidence_interval
            ax1.errorbar(x_target, result.ensemble_prediction,
                        yerr=[[result.ensemble_prediction - ci_low],
                              [ci_high - result.ensemble_prediction]],
                        color='red', fmt='none', capsize=8, linewidth=2)
        
        ax1.set_xlabel('X')
        ax1.set_ylabel('Y')
        ax1.set_title('Model Comparison')
        ax1.legend(loc='best', fontsize=8)
        ax1.grid(True, alpha=0.3)
        
        # Plot 2: Residuals
        best_model = self._models[self._best_model_name]['model']
        y_pred = best_model.evaluate(self._x_data)
        residuals = self._y_data - y_pred
        
        ax2.scatter(self._x_data, residuals,
                   color='#d32f2f', s=60, zorder=3,
                   edgecolors='white', linewidth=1)
        ax2.axhline(y=0, color='black', linestyle='--', linewidth=1, alpha=0.5)
        ax2.set_xlabel('X')
        ax2.set_ylabel('Residual')
        ax2.set_title(f'Residuals - Best Model ({self._best_model_name})')
        ax2.grid(True, alpha=0.3)
        
        # Add residual stats
        max_res = np.max(np.abs(residuals))
        mean_res = np.mean(residuals)
        std_res = np.std(residuals)
        ax2.text(0.02, 0.98, f'Max: {max_res:.2e}\nMean: {mean_res:.2e}\nStd: {std_res:.2e}',
                transform=ax2.transAxes, verticalalignment='top', fontsize=9,
                bbox=dict(boxstyle='round', facecolor='white', alpha=0.8))
        
        # Plot 3: Residual histogram
        ax3.hist(residuals, bins=20, color='#0d47a1', alpha=0.7, edgecolor='black')
        ax3.axvline(x=0, color='red', linestyle='--', linewidth=1.5)
        ax3.set_xlabel('Residual')
        ax3.set_ylabel('Frequency')
        ax3.set_title('Residual Distribution')
        ax3.grid(True, alpha=0.3)
        
        # Plot 4: Model comparison table
        ax4.axis('off')
        
        table_data = []
        headers = ['Method', 'R²', 'AIC', 'BIC', 'RMSE']
        table_data.append(headers)
        
        for name, info in self._models.items():
            row = [
                name[:20],
                f'{info["r_squared"]:.4f}',
                f'{info["aic"]:.1f}',
                f'{info["bic"]:.1f}',
                f'{info["rmse"]:.2e}'
            ]
            table_data.append(row)
        
        table = ax4.table(cellText=table_data, loc='center',
                         cellLoc='center',
                         colColours=['#0d47a1'] * len(headers),
                         rowColours=['#f8f9fa'] + ['#ffffff'] * (len(table_data)-1))
        table.auto_set_font_size(False)
        table.set_fontsize(10)
        table.scale(1, 1.8)
        
        ax4.set_title('Model Comparison', fontsize=12, pad=20)
        
        plt.tight_layout()
        return fig
    
    @property
    def is_fitted(self) -> bool:
        return self._fitted
    
    @property
    def best_model_name(self) -> str:
        return self._best_model_name
    
    @property
    def models(self) -> Dict[str, Any]:
        return self._models

# ===================================================================
# UI APPLICATION
# ===================================================================

class PredictionCalculator:
    """Advanced prediction calculator with model selection."""
    
    def __init__(self):
        self.engine = AdvancedPredictionEngine()
        self.x_inputs = []
        self.y_inputs = []
        self.pred_input = None
        self.result_output = widgets.Output()
        self.plot_output = widgets.Output()
        self._build_ui()
    
    def _get_default_data(self):
        return DEFAULT_X[:4].copy(), DEFAULT_Y[:4].copy()
    
    def _build_ui(self):
        """Build clean user interface."""
        
        container = widgets.VBox()
        
        header = widgets.HTML("""
        <div style="background: #0d47a1; padding: 20px; border-radius: 8px; text-align: center; margin-bottom: 20px;">
            <h1 style="color: #ffffff; font-size: 26px; margin: 0; font-weight: 300; letter-spacing: 1px;">
                Advanced Prediction Calculator
            </h1>
            <p style="color: #e3f2fd; font-size: 14px; margin: 5px 0 0 0;">
                Automatic model selection with uncertainty estimation
            </p>
        </div>
        """)
        
        # Data Input
        data_panel = widgets.VBox()
        
        point_count = widgets.IntSlider(
            value=4, min=3, max=10, step=1,
            description='Points:',
            style={'description_width': 'initial'},
            layout=widgets.Layout(width='300px')
        )
        
        update_btn = widgets.Button(
            description='Update',
            button_style='primary',
            layout=widgets.Layout(width='100px')
        )
        
        self.x_inputs = []
        self.y_inputs = []
        data_table = widgets.VBox()
        
        def update_table(btn):
            n = point_count.value
            self.x_inputs = []
            self.y_inputs = []
            rows = []
            
            default_x, default_y = self._get_default_data()
            
            header_row = widgets.HBox([
                widgets.Label('Point', layout=widgets.Layout(width='50px')),
                widgets.Label('X', layout=widgets.Layout(width='120px')),
                widgets.Label('Y', layout=widgets.Layout(width='120px'))
            ])
            rows.append(header_row)
            
            for i in range(n):
                x_val = default_x[i] if i < len(default_x) else default_x[-1] + (i - len(default_x) + 1) * 0.12
                y_val = default_y[i] if i < len(default_y) else default_y[-1]
                
                x = widgets.FloatText(value=float(x_val), step=0.01, layout=widgets.Layout(width='120px'))
                y = widgets.FloatText(value=float(y_val), step=0.001, layout=widgets.Layout(width='120px'))
                self.x_inputs.append(x)
                self.y_inputs.append(y)
                
                row = widgets.HBox([
                    widgets.Label(str(i+1), layout=widgets.Layout(width='50px')),
                    x, y
                ], layout=widgets.Layout(margin='2px 0'))
                rows.append(row)
            
            data_table.children = rows
        
        update_btn.on_click(update_table)
        update_table(None)
        
        data_panel.children = [
            widgets.HTML('<div style="font-size: 16px; font-weight: 600; color: #0d47a1; margin-bottom: 10px;">Data Points</div>'),
            widgets.HBox([point_count, update_btn]),
            data_table
        ]
        
        # Model Controls
        model_panel = widgets.VBox()
        
        build_btn = widgets.Button(
            description='Build Model',
            button_style='success',
            layout=widgets.Layout(width='150px', height='40px')
        )
        
        clear_btn = widgets.Button(
            description='Clear',
            button_style='danger',
            layout=widgets.Layout(width='100px', height='40px')
        )
        
        build_btn.on_click(self._build_model)
        clear_btn.on_click(self._clear)
        
        model_panel.children = [
            widgets.HTML('<div style="font-size: 16px; font-weight: 600; color: #0d47a1; margin-bottom: 10px;">Model</div>'),
            widgets.HBox([build_btn, clear_btn])
        ]
        
        # Prediction Panel
        self.prediction_panel = widgets.VBox()
        self.prediction_panel.layout.visibility = 'hidden'
        
        # Output area
        self.output_area = widgets.VBox()
        
        container.children = [
            header,
            widgets.HTML('<hr style="border: 1px solid #e0e0e0; margin: 10px 0;">'),
            data_panel,
            widgets.HTML('<hr style="border: 1px solid #e0e0e0; margin: 10px 0;">'),
            model_panel,
            widgets.HTML('<hr style="border: 1px solid #e0e0e0; margin: 10px 0;">'),
            self.prediction_panel,
            self.output_area
        ]
        
        display(HTML("""
        <style>
            .result-box {
                background: #ffffff;
                padding: 30px;
                border-radius: 8px;
                text-align: center;
                border: 2px solid #0d47a1;
                margin: 10px 0;
            }
            .result-value {
                font-size: 42px;
                font-weight: bold;
                color: #0d47a1;
                font-family: 'Courier New', monospace;
            }
            .result-label {
                font-size: 14px;
                color: #666;
                margin-bottom: 5px;
            }
            .warning-box {
                background: #fff3e0;
                padding: 10px 15px;
                border-radius: 4px;
                border-left: 4px solid #e65100;
                color: #e65100;
                margin: 10px 0;
                font-size: 13px;
            }
            .success-box {
                background: #e8f5e9;
                padding: 10px 15px;
                border-radius: 4px;
                border-left: 4px solid #2e7d32;
                color: #1b5e20;
                margin: 10px 0;
                font-size: 13px;
            }
            .details-box {
                background: #f5f5f5;
                padding: 15px;
                border-radius: 4px;
                margin: 10px 0;
                font-size: 13px;
                color: #555;
                border: 1px solid #e0e0e0;
            }
            .details-box table {
                width: 100%;
                border-collapse: collapse;
            }
            .details-box td {
                padding: 4px 10px;
                border-bottom: 1px solid #e8e8e8;
            }
            .details-box td:first-child {
                font-weight: 600;
                color: #333;
                width: 40%;
            }
            .input-row {
                display: flex;
                gap: 15px;
                align-items: center;
                flex-wrap: wrap;
                margin: 10px 0;
            }
            .collapsible {
                cursor: pointer;
                user-select: none;
                color: #0d47a1;
                font-weight: 500;
            }
            .collapsible:hover {
                color: #1565c0;
            }
            .confidence-bar {
                width: 100%;
                height: 20px;
                background: #e0e0e0;
                border-radius: 10px;
                overflow: hidden;
                margin: 5px 0;
            }
            .confidence-bar-fill {
                height: 100%;
                border-radius: 10px;
                transition: width 0.5s;
            }
        </style>
        """))
        
        display(container)
    
    def _get_data(self):
        x = np.array([w.value for w in self.x_inputs])
        y = np.array([w.value for w in self.y_inputs])
        return x, y
    
    def _format_confidence_bar(self, std: float, pred: float) -> str:
        """Format confidence bar based on relative uncertainty."""
        if abs(pred) < MACHINE_EPSILON:
            return '<div class="confidence-bar"><div class="confidence-bar-fill" style="width: 0%; background: #666;"></div></div>'
        
        rel_std = std / abs(pred)
        confidence = max(0, min(100, 100 * (1 - rel_std)))
        
        if confidence >= 90:
            color = '#2e7d32'
            level = 'High'
        elif confidence >= 70:
            color = '#388e3c'
            level = 'Good'
        elif confidence >= 50:
            color = '#f57c00'
            level = 'Moderate'
        elif confidence >= 30:
            color = '#d32f2f'
            level = 'Low'
        else:
            color = '#b71c1c'
            level = 'Very Low'
        
        return f"""
        <div>
            <div style="display: flex; justify-content: space-between; font-size: 13px; color: #666;">
                <span>Confidence: {confidence:.0f}%</span>
                <span>{level}</span>
            </div>
            <div class="confidence-bar">
                <div class="confidence-bar-fill" style="width: {confidence}%; background: {color};"></div>
            </div>
        </div>
        """
    
    def _format_result(self, result: PredictionResult) -> widgets.HTML:
        """Display prediction result."""
        
        # Main result
        html = f"""
        <div class="result-box">
            <div class="result-label">Ensemble Prediction</div>
            <div class="result-value">{result.ensemble_prediction:.10f}</div>
            <div style="margin-top: 10px; font-size: 14px; color: #666;">
                ± {result.ensemble_std:.6f} (std deviation)
            </div>
        </div>
        
        <div style="margin: 15px 0; padding: 15px; background: #f8f9fa; border-radius: 8px; border: 1px solid #e0e0e0;">
            {self._format_confidence_bar(result.ensemble_std, result.ensemble_prediction)}
        </div>
        """
        
        # Prediction intervals
        html += f"""
        <div style="display: grid; grid-template-columns: 1fr 1fr; gap: 15px; margin: 10px 0;">
            <div style="background: #e8f0fe; padding: 15px; border-radius: 4px; border: 1px solid #90caf9;">
                <div style="font-size: 12px; color: #666;">95% Confidence Interval</div>
                <div style="font-size: 16px; font-weight: bold; color: #0d47a1;">
                    [{result.confidence_interval[0]:.6f}, {result.confidence_interval[1]:.6f}]
                </div>
            </div>
            <div style="background: #e8f0fe; padding: 15px; border-radius: 4px; border: 1px solid #90caf9;">
                <div style="font-size: 12px; color: #666;">95% Prediction Interval</div>
                <div style="font-size: 16px; font-weight: bold; color: #0d47a1;">
                    [{result.prediction_interval[0]:.6f}, {result.prediction_interval[1]:.6f}]
                </div>
            </div>
        </div>
        """
        
        # Best method
        if result.best_method in result.methods:
            best = result.methods[result.best_method]
            html += f"""
            <div style="margin: 10px 0; padding: 10px; background: #e8f5e9; border-radius: 4px; border-left: 4px solid #2e7d32;">
                <div style="font-size: 13px; font-weight: 600; color: #1b5e20;">Best Model: {result.best_method}</div>
                <div style="font-size: 12px; color: #555;">R²: {best.r_squared:.6f} | RMSE: {best.rmse:.2e}</div>
            </div>
            """
        
        # Warning
        if result.is_extrapolation:
            html += f"""
            <div class="warning-box">
                Warning: {result.warning or 'Value is outside the data range'}
            </div>
            """
        
        # Method comparison table
        html += """
        <details style="margin-top: 15px;">
            <summary class="collapsible">Method Comparison</summary>
            <div class="details-box">
                <table>
                    <thead>
                        <tr>
                            <th>Method</th>
                            <th>Prediction</th>
                            <th>R²</th>
                            <th>RMSE</th>
                            <th>AIC</th>
                        </tr>
                    </thead>
                    <tbody>
        """
        
        for name, model in result.methods.items():
            html += f"""
                        <tr>
                            <td>{name}</td>
                            <td>{model.prediction:.6f}</td>
                            <td>{model.r_squared:.4f}</td>
                            <td>{model.rmse:.2e}</td>
                            <td>{model.aic:.1f}</td>
                        </tr>
            """
        
        html += """
                    </tbody>
                </table>
            </div>
        </details>
        """
        
        # Technical details
        html += f"""
        <details style="margin-top: 10px;">
            <summary class="collapsible">Technical Details</summary>
            <div class="details-box">
                <table>
                    <tr><td>X Target</td><td>{result.x_target:.6f}</td></tr>
                    <tr><td>Extrapolation Distance</td><td>{result.extrapolation_distance:.2f}x</td></tr>
                    <tr><td>Number of Methods</td><td>{len(result.methods)}</td></tr>
                    <tr><td>Ensemble Std Dev</td><td>{result.ensemble_std:.2e}</td></tr>
                    <tr><td>Execution Time</td><td>{result.execution_time*1000:.2f}ms</td></tr>
                </table>
            </div>
        </details>
        """
        
        return widgets.HTML(html)
    
    def _build_model(self, btn):
        """Build the prediction model."""
        self.output_area.children = []
        
        x_data, y_data = self._get_data()
        
        if np.any(np.isnan(x_data)) or np.any(np.isnan(y_data)):
            self.output_area.children = [
                widgets.HTML("""
                <div style="padding: 15px; background: #fff3e0; border-radius: 4px; border-left: 4px solid #e65100;">
                    Please enter valid numeric values for all data points.
                </div>
                """)
            ]
            return
        
        try:
            fit_results = self.engine.fit(x_data, y_data)
        except Exception as e:
            self.output_area.children = [
                widgets.HTML(f"""
                <div style="padding: 15px; background: #ffcdd2; border-radius: 4px; border-left: 5px solid #b71c1c;">
                    Error: {str(e)}
                </div>
                """)
            ]
            return
        
        # Success message
        best = fit_results['best_model']
        n_models = len(fit_results['models'])
        success_msg = widgets.HTML(f"""
        <div class="success-box">
            <b>Model built successfully</b><br>
            {n_models} models evaluated | Best: {best} | {fit_results['n_points']} data points
        </div>
        """)
        
        # Prediction UI
        pred_label = widgets.HTML("""
        <div style="font-size: 16px; font-weight: 600; color: #0d47a1; margin: 15px 0 10px 0;">
            Estimate Y from X
        </div>
        """)
        
        self.pred_input = widgets.FloatText(
            value=float(np.mean(x_data)),
            step=0.01,
            layout=widgets.Layout(width='180px')
        )
        
        predict_btn = widgets.Button(
            description='Predict',
            button_style='success',
            layout=widgets.Layout(width='120px', height='36px')
        )
        
        plot_btn = widgets.Button(
            description='Visualize',
            button_style='primary',
            layout=widgets.Layout(width='120px', height='36px')
        )
        
        self.result_output = widgets.Output()
        
        def on_predict(btn):
            with self.result_output:
                clear_output(wait=True)
                x_target = self.pred_input.value
                try:
                    result = self.engine.predict(x_target)
                    display(self._format_result(result))
                except Exception as e:
                    display(HTML(f"""
                    <div style="padding: 10px; background: #ffcdd2; border-radius: 4px; color: #b71c1c;">
                        Error: {str(e)}
                    </div>
                    """))
        
        def on_plot(btn):
            with self.plot_output:
                clear_output(wait=True)
                try:
                    x_target = self.pred_input.value if self.pred_input is not None else None
                    fig = self.engine.plot(x_target)
                    plt.close(fig)
                    display(fig)
                except Exception as e:
                    display(HTML(f"""
                    <div style="padding: 10px; background: #ffcdd2; border-radius: 4px; color: #b71c1c;">
                        Plot generation failed: {str(e)}
                    </div>
                    """))
        
        predict_btn.on_click(on_predict)
        plot_btn.on_click(on_plot)
        
        input_row = widgets.HBox([
            widgets.Label('Enter X value:', layout=widgets.Layout(width='120px')),
            self.pred_input,
            predict_btn,
            plot_btn
        ], layout=widgets.Layout(margin='10px 0'))
        
        self.plot_output = widgets.Output()
        
        self.prediction_panel.children = [
            pred_label,
            input_row,
            self.result_output,
            self.plot_output
        ]
        self.prediction_panel.layout.visibility = 'visible'
        
        self.output_area.children = [success_msg]
        on_predict(None)
    
    def _clear(self, btn):
        """Clear all output."""
        self.prediction_panel.children = []
        self.prediction_panel.layout.visibility = 'hidden'
        self.output_area.children = [
            widgets.HTML("""
            <div style="padding: 20px; text-align: center; color: #666; background: #f8f9fa; border-radius: 8px;">
                Ready. Enter data and click "Build Model" to begin.
            </div>
            """)
        ]
        self.pred_input = None

# ===================================================================
# ENTRY POINT
# ===================================================================

if __name__ == "__main__":
    print("\n" + "=" * 70)
    print("ADVANCED PREDICTION CALCULATOR")
    print("Scientific Computing Implementation")
    print("=" * 70)
    print("\nFeatures:")
    print("  - Automatic model selection (AIC, BIC)")
    print("  - Multiple interpolation methods")
    print("  - Prediction intervals (95% confidence)")
    print("  - Bootstrap uncertainty estimation")
    print("  - Residual analysis")
    print("  - Model comparison dashboard")
    print("  - Clean, minimal interface")
    print("\nInitialization completed.")
    print("=" * 70 + "\n")
    
    app = PredictionCalculator()


PREDICTION CALCULATOR WITH CONFIDENCE SCORING
Scientific Computing Implementation

Features:
  - Clean, minimal interface
  - Automatic interpolation/extrapolation detection
  - Prediction confidence scoring (0-100%)
  - Confidence reasons and transparency
  - Calculation steps display
  - Technical details available on demand
  - Default data pre-loaded
  - Visualization on request

Initialization completed.

